In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
path = "/content/drive/MyDrive/BATTERY_SOC_PROJECT/SOC DATA SET"

In [3]:
import os
path = "/content/drive/MyDrive/BATTERY_SOC_PROJECT/SOC DATA SET"

In [5]:
import joblib
import os

soc_path = "/content/drive/MyDrive/BATTERY_SOC_PROJECT/day22_outputs/FINAL_SOC_XGBoost_day22.joblib"
remaining_path = "/content/drive/MyDrive/BATTERY_SOC_PROJECT/day22_outputs/FINAL_Remaining_Charging_Time_Random_Forest_day22.joblib"

soc_artifact = joblib.load(soc_path)
remaining_artifact = joblib.load(remaining_path)

soc_model = soc_artifact["model"]
remaining_model = remaining_artifact["model"]
print("Type:", type(soc_model))
print("Features:", soc_model.feature_names_in_)
print("Number of features:", len(soc_model.feature_names_in_))
print("Type:", type(remaining_model))
print("Features:", remaining_model.feature_names_in_)
print("Number of features:", len(remaining_model.feature_names_in_))

Type: <class 'xgboost.sklearn.XGBRegressor'>
Features: ['Voltage' 'Current' 'Temperature' 'Elapsed Time' 'Cycle' 'WhAccu' 'Power']
Number of features: 7
Type: <class 'sklearn.ensemble._forest.RandomForestRegressor'>
Features: ['Voltage' 'Current' 'Temperature' 'Elapsed Time' 'Cycle' 'WhAccu' 'Power']
Number of features: 7


In [6]:
import os
import json
import shutil
import zipfile
from datetime import datetime
import joblib
import pandas as pd
import numpy as np
BASE_PATH = "/content/drive/MyDrive/BATTERY_SOC_PROJECT"
DAY22_PATH = os.path.join(BASE_PATH, "day22_outputs")
DAY23_PATH = os.path.join(BASE_PATH, "day23_outputs")
DAY24_PATH = os.path.join(BASE_PATH, "day24_outputs")
DAY25_PATH = os.path.join(BASE_PATH, "day25_outputs")
DAY26_PATH = os.path.join(BASE_PATH, "day26_outputs")
PACKAGE_PATH = os.path.join(DAY26_PATH, "FINAL_SOFTWARE_PACKAGE")
SOC_MODEL_PATH = os.path.join(DAY22_PATH,"FINAL_SOC_XGBoost_day22.joblib")
REMAINING_MODEL_PATH = os.path.join(DAY22_PATH,"FINAL_Remaining_Charging_Time_Random_Forest_day22.joblib")
if os.path.exists(PACKAGE_PATH):
    shutil.rmtree(PACKAGE_PATH)
folders = [
    "models","configuration","pipeline","examples","validation_records","analysis","figures","tables","documentation"]
for folder in folders:
    os.makedirs(os.path.join(PACKAGE_PATH, folder), exist_ok=True)
print("DAY 26 FINAL SOFTWARE PACKAGE")
print("Package path:")
print(PACKAGE_PATH)
print()
print("Loading frozen Day 22 models...")
soc_artifact = joblib.load(SOC_MODEL_PATH)
remaining_artifact = joblib.load(REMAINING_MODEL_PATH)
soc_model = soc_artifact["model"]
remaining_model = remaining_artifact["model"]
print("SOC model:", type(soc_model))
print("Remaining Charging Time model:", type(remaining_model))
print()
SOC_FEATURES = list(soc_model.feature_names_in_)
REMAINING_FEATURES = list(remaining_model.feature_names_in_)
print("Actual SOC model features:")
print(SOC_FEATURES)
print()
print("Actual Remaining Charging Time model features:")
print(REMAINING_FEATURES)
print()
if SOC_FEATURES != REMAINING_FEATURES:
    raise ValueError("The two frozen models do not use the same feature list. ""Check model artifacts before continuing.")
FINAL_FEATURES = SOC_FEATURES
print("FINAL VERIFIED FEATURE LIST:")
for i, feature in enumerate(FINAL_FEATURES, 1):
    print(f"{i}. {feature}")
print()
shutil.copy2(
    SOC_MODEL_PATH,
    os.path.join(PACKAGE_PATH,"models","FINAL_SOC_XGBoost_day22.joblib"))
shutil.copy2(
    REMAINING_MODEL_PATH,
    os.path.join(PACKAGE_PATH,"models","FINAL_Remaining_Charging_Time_Random_Forest_day22.joblib"))
print("Frozen models copied successfully.")
print()
feature_config = {
    "project": "Battery State of Charge and Remaining Charging Time Prediction",
    "final_features": FINAL_FEATURES,
    "SOC_features": SOC_FEATURES,
    "Remaining_Charging_Time_features": REMAINING_FEATURES,
    "excluded_from_final_model_inputs": ["Capacity"],
    "note": ("Feature lists were read directly from the frozen Day 22 ""model artifacts. No retraining was performed.")}
feature_config_path = os.path.join(
    PACKAGE_PATH,"configuration","final_feature_configuration.json")
with open(feature_config_path, "w") as f:json.dump(feature_config, f, indent=4)
model_selection = {
    "SOC": {
        "task": "Battery State of Charge Prediction",
        "model": "XGBoost Regressor",
        "artifact": "FINAL_SOC_XGBoost_day22.joblib"
    },
    "Remaining Charging Time": {
        "task": "Remaining Charging Time Prediction",
        "model": "Random Forest Regressor",
        "artifact": "FINAL_Remaining_Charging_Time_Random_Forest_day22.joblib"
    }
}

model_selection_path = os.path.join(
    PACKAGE_PATH,
    "configuration",
    "model_selection.json"
)

with open(model_selection_path, "w") as f:
    json.dump(model_selection, f, indent=4)

print("Feature and model configuration saved.")
print()

DAY 26 FINAL SOFTWARE PACKAGE
Package path:
/content/drive/MyDrive/BATTERY_SOC_PROJECT/day26_outputs/FINAL_SOFTWARE_PACKAGE

Loading frozen Day 22 models...
SOC model: <class 'xgboost.sklearn.XGBRegressor'>
Remaining Charging Time model: <class 'sklearn.ensemble._forest.RandomForestRegressor'>

Actual SOC model features:
[np.str_('Voltage'), np.str_('Current'), np.str_('Temperature'), np.str_('Elapsed Time'), np.str_('Cycle'), np.str_('WhAccu'), np.str_('Power')]

Actual Remaining Charging Time model features:
['Voltage', 'Current', 'Temperature', 'Elapsed Time', 'Cycle', 'WhAccu', 'Power']

FINAL VERIFIED FEATURE LIST:
1. Voltage
2. Current
3. Temperature
4. Elapsed Time
5. Cycle
6. WhAccu
7. Power

Frozen models copied successfully.

Feature and model configuration saved.



In [7]:
records_to_copy = [(DAY22_PATH, "DAY22_FREEZE_RECORD.json"),(DAY23_PATH, "DAY23_VALIDATION_RECORD.json"),(DAY24_PATH, "DAY24_TECHNICAL_REVIEW_RECORD.json")]
for source_folder, filename in records_to_copy:
    source = os.path.join(source_folder, filename)
    if os.path.exists(source):
        shutil.copy2(source, os.path.join(PACKAGE_PATH, "validation_records", filename ))
        print("Copied:", filename)

Copied: DAY22_FREEZE_RECORD.json
Copied: DAY23_VALIDATION_RECORD.json
Copied: DAY24_TECHNICAL_REVIEW_RECORD.json


In [8]:
if os.path.exists(DAY25_PATH):
    for filename in os.listdir(DAY25_PATH):
        if filename.endswith((".csv", ".json", ".txt")):
            source = os.path.join(DAY25_PATH,filename)
            destination = os.path.join(PACKAGE_PATH,"analysis",filename)
            shutil.copy2(source, destination)
            print("Copied Day 25 analysis:", filename)
print()

Copied Day 25 analysis: DAY25_ERROR_STATISTICS.csv
Copied Day 25 analysis: DAY25_FAILURE_MODE_ANALYSIS.csv
Copied Day 25 analysis: DAY25_ERROR_FAILURE_REVIEW_RECORD.json
Copied Day 25 analysis: DAY25_SUBMISSION_NOTE.txt



In [9]:
example_input = {"Voltage": 3.85,"Current": -1.50,"Temperature": 25.0,"Elapsed Time": 1800.0,"Cycle": 50,"WhAccu": 8.50,"Power": -5.775}
example_df = pd.DataFrame(
    [example_input],
    columns=FINAL_FEATURES)
example_path = os.path.join(PACKAGE_PATH,"examples","example_input.csv")
example_df.to_csv(
    example_path,
    index=False)
print("Example input created.")
print(example_df)
print()

Example input created.
   Voltage  Current  Temperature  Elapsed Time  Cycle  WhAccu  Power
0     3.85     -1.5         25.0        1800.0     50     8.5 -5.775



In [10]:
print("Running representative inference test...")
X_example = example_df[FINAL_FEATURES]
soc_prediction = soc_model.predict(X_example)
remaining_prediction = remaining_model.predict(X_example)
print()
print("Example SOC prediction:")
print(float(soc_prediction[0]))
print()
print("Example Remaining Charging Time prediction:")
print(float(remaining_prediction[0]))
print()


Running representative inference test...

Example SOC prediction:
59.56431198120117

Example Remaining Charging Time prediction:
135.61408713808055



/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


In [11]:
inference_code = r'''
import os
import joblib
import pandas as pd
# Model paths
PACKAGE_DIR = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
SOC_MODEL_PATH = os.path.join(
    PACKAGE_DIR,
    "models",
    "FINAL_SOC_XGBoost_day22.joblib"
)
REMAINING_MODEL_PATH = os.path.join(
    PACKAGE_DIR,
    "models",
    "FINAL_Remaining_Charging_Time_Random_Forest_day22.joblib"
)
# Load frozen models
soc_artifact = joblib.load(SOC_MODEL_PATH)
remaining_artifact = joblib.load(REMAINING_MODEL_PATH)
soc_model = soc_artifact["model"]
remaining_model = remaining_artifact["model"]
# Read feature names directly from models
SOC_FEATURES = list(soc_model.feature_names_in_)
REMAINING_FEATURES = list(
    remaining_model.feature_names_in_
)
# SOC prediction
def predict_soc(input_data):
    if isinstance(input_data, dict):
        input_data = pd.DataFrame([input_data])
    X = input_data[SOC_FEATURES]
    prediction = soc_model.predict(X)
    return prediction
# Remaining Charging Time prediction
def predict_remaining_charging_time(input_data):
    if isinstance(input_data, dict):
        input_data = pd.DataFrame([input_data])
    X = input_data[REMAINING_FEATURES]
    prediction = remaining_model.predict(X)
    return prediction
# Combined prediction
def predict_battery_status(input_data):
    if isinstance(input_data, dict):
        input_data = pd.DataFrame([input_data])
    soc = predict_soc(input_data)
    remaining_time = predict_remaining_charging_time(
        input_data
    )
    return {
        "SOC_prediction": float(soc[0]),
        "Remaining_Charging_Time_prediction": float(
            remaining_time[0])}
# Example
if __name__ == "__main__":
    sample = {
        "Voltage": 3.85,
        "Current": -1.50,
        "Temperature": 25.0,
        "Elapsed Time": 1800.0,
        "Cycle": 50,
        "WhAccu": 8.50,
        "Power": -5.775
    }
    result = predict_battery_status(sample)
    print("Battery Prediction")
    print("------------------")
    print("SOC:",
          result["SOC_prediction"])
    print(
        "Remaining Charging Time:",
        result[
            "Remaining_Charging_Time_prediction"
        ]
    )
'''
inference_path = os.path.join(
    PACKAGE_PATH,
    "pipeline",
    "inference_workflow.py")
with open(inference_path, "w") as f:
    f.write(inference_code)
print("Inference workflow created.")
print()

Inference workflow created.



In [12]:
pipeline_text = """
BATTERY PREDICTION PIPELINE
The final software package uses the frozen Day 22 models.
Tasks
1. Battery State of Charge (SOC) Prediction
   Model: XGBoost Regressor
2. Remaining Charging Time Prediction
   Model: Random Forest Regressor
Final Model Inputs
Both frozen models use the following 7 features:
1. Voltage
2. Current
3. Temperature
4. Elapsed Time
5. Cycle
6. WhAccu
7. Power

Capacity
Capacity is NOT an input feature of the frozen final models.
The feature list was obtained directly from the saved model
artifacts using the model's feature_names_in_ property.


Inference
A new battery observation is supplied using the seven required
features.
The SOC model produces the SOC prediction.
The Remaining Charging Time model produces the remaining-time
prediction.


Important
The final frozen models are used directly.

No retraining is performed during inference.
"""
pipeline_path = os.path.join(
    PACKAGE_PATH,
    "pipeline",
    "PIPELINE_DESCRIPTION.txt"
)
with open(pipeline_path, "w") as f:
    f.write(pipeline_text)

In [13]:
results_table = pd.DataFrame({
    "Task": ["SOC Prediction","Remaining Charging Time Prediction","Remaining Charging Time Validation"],
    "Final Model": ["XGBoost Regressor","Random Forest Regressor","Random Forest Regressor"],
    "R2": [0.990521,0.998756,0.996502]})
results_table_path = os.path.join(PACKAGE_PATH,"tables","FINAL_RESULTS_TABLE.csv")
results_table.to_csv(results_table_path,index=False)
print("Final results table created.")
print(results_table)
print()

Final results table created.
                                 Task              Final Model        R2
0                      SOC Prediction        XGBoost Regressor  0.990521
1  Remaining Charging Time Prediction  Random Forest Regressor  0.998756
2  Remaining Charging Time Validation  Random Forest Regressor  0.996502



In [14]:
result_summary = """
FINAL MODEL RESULT SUMMARY
SOC Prediction
Model: XGBoost Regressor
Test R²: 0.990521
Remaining Charging Time Prediction
Model: Random Forest Regressor
Test R²: 0.998756
Remaining Charging Time Validation
Model: Random Forest Regressor
Validation R²: 0.996502
Final Feature Set
1. Voltage
2. Current
3. Temperature
4. Elapsed Time
5. Cycle
6. WhAccu
7. Power
Important:
Capacity is not an input feature of the frozen final models.
Model Selection
SOC:
XGBoost Regressor
Remaining Charging Time:
Random Forest Regressor
The reported values are based on the existing frozen
validation/test evidence. No retraining was performed during
Day 26.
"""
summary_path = os.path.join(PACKAGE_PATH,"documentation","FINAL_RESULT_SUMMARY.txt")
with open(summary_path, "w") as f:
    f.write(result_summary)


In [15]:
readme = """
# Battery SOC and Remaining Charging Time Prediction
## Project Overview
This project develops machine learning models for:
1. Battery State of Charge (SOC) prediction
2. Remaining Charging Time prediction
The final software package contains the frozen models,
configuration files, inference workflow, validation records,
analysis files and documentation.
## Final Models
### SOC Prediction
Model:
XGBoost Regressor
Test R²:
0.990521
### Remaining Charging Time Prediction
Model:
Random Forest Regressor
Test R²:
0.998756
### Remaining Charging Time Validation
Validation R²:
0.996502
## Final Input Features
Both final frozen models use the same seven features:
- Voltage
- Current
- Temperature
- Elapsed Time
- Cycle
- WhAccu
- Power
Capacity is not an input feature of the frozen final models.
## Package Structure
models/
    Final frozen model files
configuration/
    Feature configuration
    Model selection configuration
pipeline/
    Inference workflow
    Pipeline description
examples/
    Example input
validation_records/
    Day 22 freeze record
    Day 23 validation record
    Day 24 technical review record
analysis/
    Day 25 error and failure analysis
tables/
    Final results table
documentation/
    Final result summary
    Report
    Presentation draft
## Running the Inference Workflow
Required packages:
- Python
- pandas
- numpy
- scikit-learn
- xgboost
- joblib
Run:
python pipeline/inference_workflow.py
The workflow loads the frozen models and produces:
- SOC prediction
- Remaining Charging Time prediction
## Important
The final software package uses the frozen Day 22 models.
No model retraining is performed during inference.
The model feature lists were verified directly from the frozen
model artifacts.
"""
readme_path = os.path.join(PACKAGE_PATH,"README.md")
with open(readme_path, "w") as f:
    f.write(readme)

In [16]:
setup_text = """
SETUP AND RUN INSTRUCTIONS
1. Install Python.
2. Install required libraries:
   pip install pandas numpy scikit-learn xgboost joblib
3. Keep the following package structure unchanged:
   models/
   configuration/
   pipeline/
   examples/
   validation_records/
   analysis/
   tables/
   documentation/
4. Run:
   python pipeline/inference_workflow.py
5. The program loads the frozen models and generates:
   - SOC prediction
   - Remaining Charging Time prediction
The example input is available in:
examples/example_input.csv
"""
setup_path = os.path.join(PACKAGE_PATH,"documentation","SETUP_AND_RUN_INSTRUCTIONS.txt")
with open(setup_path, "w") as f:
    f.write(setup_text)

In [17]:
report_text = """
FINAL PROJECT REPORT DRAFT
Battery State of Charge and Remaining Charging Time
Prediction Using Machine Learning
1. INTRODUCTION
Battery monitoring is important for understanding the present
state of a battery and estimating how much charging time remains.
This project develops machine learning models for two related
prediction tasks:
1. State of Charge (SOC) prediction
2. Remaining Charging Time prediction
2. OBJECTIVES
The main objectives are:
- Prepare battery data for machine learning.
- Develop an SOC prediction model.
- Develop a Remaining Charging Time prediction model.
- Compare and validate the developed models.
- Analyze prediction errors and possible failure cases.
- Freeze the final models.
- Create a software package for inference.
3. INPUT FEATURES
The final frozen models use seven input features:
- Voltage
- Current
- Temperature
- Elapsed Time
- Cycle
- WhAccu
- Power
Capacity is not an input feature of the final frozen models.
4. FINAL MODEL SELECTION
For SOC prediction, the frozen final model is:
XGBoost Regressor
For Remaining Charging Time prediction, the frozen final
model is:
Random Forest Regressor
5. RESULTS
SOC Prediction:
XGBoost Regressor
Test R² = 0.990521
Remaining Charging Time:
Random Forest Regressor
Test R² = 0.998756
Remaining Charging Time Validation:
Random Forest Regressor
Validation R² = 0.996502
6. ERROR ANALYSIS
The project includes an error and failure analysis stage.
Potential sources of prediction error include:
- High-error observations
- Charging-phase transitions
- Temperature variation
- Cycle-to-cycle variation
- Sensitivity of remaining-time prediction to changing
  charging conditions
These observations are used for technical review and future
improvement.
7. SOFTWARE PACKAGE
The final package contains:
- Frozen SOC model
- Frozen Remaining Charging Time model
- Feature configuration
- Model selection configuration
- Inference workflow
- Example input
- Validation records
- Error analysis
- Results table
- README
- Setup instructions
8. CONCLUSION
The project produced separate final machine learning models
for SOC and Remaining Charging Time prediction.
The frozen SOC XGBoost model achieved a test R² of 0.990521.
The frozen Remaining Charging Time Random Forest model achieved
a test R² of 0.998756.
The final software package provides a reproducible inference
workflow using the frozen models.
"""
report_path = os.path.join(PACKAGE_PATH,"documentation","FINAL_REPORT_DRAFT.txt")
with open(report_path, "w") as f:
    f.write(report_text)

In [18]:
day26_record = {
    "day": 26,
    "title": "Software Package and Final Documentation",
    "timestamp": datetime.now().isoformat(),
    "retraining_performed": False,
    "models": {
        "SOC": {
            "model": "XGBoost Regressor",
            "artifact": "FINAL_SOC_XGBoost_day22.joblib",
            "features": SOC_FEATURES,
            "test_R2": 0.990521
        },
        "Remaining Charging Time": {
            "model": "Random Forest Regressor",
            "artifact": "FINAL_Remaining_Charging_Time_Random_Forest_day22.joblib",
            "features": REMAINING_FEATURES,
            "test_R2": 0.998756,
            "validation_R2": 0.996502
        }
    },
    "package_structure_created": True,
    "inference_test_completed": True,
    "capacity_used_as_model_input": False,
    "notes": [
        "Final feature lists were read directly from frozen model artifacts.",
        "Both final models use seven features.",
        "No retraining was performed.",
        "Day 22-25 records were preserved."
    ]
}
day26_record_path = os.path.join(
    PACKAGE_PATH,
    "validation_records",
    "DAY26_SOFTWARE_PACKAGE_RECORD.json"
)
with open(day26_record_path, "w") as f:
    json.dump(day26_record, f, indent=4)

In [19]:
submission_note = """
DAY 26 SUBMISSION NOTE
Day 26 software packaging and documentation completed.
Completed:
[✓] Frozen SOC model packaged
[✓] Frozen Remaining Charging Time model packaged
[✓] Actual model feature list verified
[✓] Inference workflow created
[✓] Example input created
[✓] Results table created
[✓] README created
[✓] Setup and run instructions created
[✓] Final report draft created
[✓] Presentation draft created
[✓] Validation records preserved
[✓] Day 25 analysis preserved
[✓] Software package ZIP created
Final feature list:
Voltage
Current
Temperature
Elapsed Time
Cycle
WhAccu
Power
No retraining was performed during Day 26.
"""
submission_note_path = os.path.join(
    DAY26_PATH,
    "DAY26_SUBMISSION_NOTE.txt"
)
with open(submission_note_path, "w") as f:
    f.write(submission_note)


In [20]:
zip_path = os.path.join(
    DAY26_PATH,
    "FINAL_SOFTWARE_PACKAGE_DAY26.zip"
)
if os.path.exists(zip_path):
    os.remove(zip_path)
with zipfile.ZipFile(
    zip_path,
    "w",
    zipfile.ZIP_DEFLATED
) as zipf:
    for root, dirs, files in os.walk(PACKAGE_PATH):
        for file in files:
            full_path = os.path.join(root,file)
            relative_path = os.path.relpath(
                full_path,
                PACKAGE_PATH
            )
            zipf.write(
                full_path,
                os.path.join(
                    "FINAL_SOFTWARE_PACKAGE",
                    relative_path))

In [22]:
print()
print("DAY 26 COMPLETED SUCCESSFULLY")
print()
print("Final verified features:")
for feature in FINAL_FEATURES:
    print(" ✓", feature)
print()
print("SOC model:")
print("  XGBoost Regressor")
print("  Test R² =", 0.990521)
print()
print("Remaining Charging Time model:")
print("  Random Forest Regressor")
print("  Test R² =", 0.998756)
print("  Validation R² =", 0.996502)
print()
print("Capacity used as model input: NO")
print()
print("Package:")
print(PACKAGE_PATH)
print()
print("ZIP:")
print(zip_path)
print()
print("Example SOC prediction:",
      float(soc_prediction[0]))
print(
    "Example Remaining Charging Time prediction:",
    float(remaining_prediction[0])
)
print()
print("IMPORTANT: NO RETRAINING WAS PERFORMED.")
print("DAY 22–25 MODELS AND VALIDATION RECORDS WERE NOT CHANGED.")


DAY 26 COMPLETED SUCCESSFULLY

Final verified features:
 ✓ Voltage
 ✓ Current
 ✓ Temperature
 ✓ Elapsed Time
 ✓ Cycle
 ✓ WhAccu
 ✓ Power

SOC model:
  XGBoost Regressor
  Test R² = 0.990521

Remaining Charging Time model:
  Random Forest Regressor
  Test R² = 0.998756
  Validation R² = 0.996502

Capacity used as model input: NO

Package:
/content/drive/MyDrive/BATTERY_SOC_PROJECT/day26_outputs/FINAL_SOFTWARE_PACKAGE

ZIP:
/content/drive/MyDrive/BATTERY_SOC_PROJECT/day26_outputs/FINAL_SOFTWARE_PACKAGE_DAY26.zip

Example SOC prediction: 59.56431198120117
Example Remaining Charging Time prediction: 135.61408713808055

IMPORTANT: NO RETRAINING WAS PERFORMED.
DAY 22–25 MODELS AND VALIDATION RECORDS WERE NOT CHANGED.
